<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule: score content 80 (fix_ctr) if it ranks on page 1 (position ≤ 10) but has CTR below 0.5% — a threshold chosen from Signal 1's own bucket averages (position 4–10 averages ~0.49% CTR, so anything meaningfully under that is underperforming its rank). Otherwise, score 20 (monitor). One reason code throughout: low_ctr_high_position.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import pandas as pd
import numpy as np
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
# Load March 2026 fact table
fact_cols = [
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_data_available", "ga4_data_available",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]
panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    columns=fact_cols, storage_options={"token": HF_TOKEN}
)

# Load content dimension table
dim_cols = ["client_hash_id", "content_hash_id", "content_type"]
dim_content = pd.read_parquet(
    f"{HF_BASE}/dim_content.parquet",
    columns=dim_cols, storage_options={"token": HF_TOKEN}
)

for col in ["client_hash_id", "content_hash_id"]:
    panel_daily[col] = panel_daily[col].astype("category")
    dim_content[col] = dim_content[col].astype("category")

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")
panel_daily = panel_daily.merge(dim_content_clean, on=["client_hash_id","content_hash_id"], how="left")

print("Loaded shape:", panel_daily.shape)

# Aggregate to content-level
content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"),
    gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"),
    ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first"),
).reset_index()

# Ratios
content_level["ctr"] = content_level["gsc_clicks"] / content_level["gsc_impressions"].replace(0, np.nan)
content_level["engagement_rate"] = content_level["ga4_engaged_sessions"] / content_level["gsc_clicks"].replace(0, np.nan)

print("Content-level shape:", content_level.shape)
content_level.head()

Loaded shape: (9841378, 13)
Content-level shape: (331437, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,content_type,ctr,engagement_rate
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,keyword article,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,keyword article,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,keyword article,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,keyword article,0.0,NaN
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,keyword article,NaN,NaN


Signal 1 (flag-linked): CTR vs. position. Verdict: CONFIRMED.

Signal 2: volume vs. engagement. Verdict: OPPOSITE.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1 (flag-linked): CTR vs position
position_bins = pd.cut(content_level["gsc_avg_position"], bins=[0,3,10,20,100,1000])
signal1_table = content_level.groupby(position_bins, observed=True).agg(n=("ctr","count"), avg_ctr=("ctr","mean"))
print(signal1_table)

# Signal 2: volume — behind quick-win logic
volume_bins = pd.qcut(content_level["gsc_impressions"], q=4, duplicates="drop")
signal2_table = content_level.groupby(volume_bins, observed=True).agg(n=("engagement_rate","count"), avg_engagement=("engagement_rate","mean"))
print(signal2_table)

                      n   avg_ctr
gsc_avg_position                 
(0, 3]            16144  0.010589
(3, 10]           81988  0.004926
(10, 20]          32203  0.003211
(20, 100]         44867  0.001918
(100, 1000]         102  0.006127
                       n  avg_engagement
gsc_impressions                         
(-0.001, 2.0]        313        0.057508
(2.0, 216.0]        9182        0.057891
(216.0, 617124.0]  59342        0.046041


In [16]:
CTR_THRESHOLD = 0.005  # 0.5%

def score_row(row):
    if row["gsc_impressions"] == 0:
        return 0, "low_ctr_high_position", "monitor"
    if pd.notna(row["ctr"]) and row["gsc_avg_position"] <= 10 and row["ctr"] < CTR_THRESHOLD:
        return 80, "low_ctr_high_position", "fix_ctr"
    return 20, "low_ctr_high_position", "monitor"

content_level[["score","reason_code","action"]] = content_level.apply(lambda r: pd.Series(score_row(r)), axis=1)

queue = content_level.sort_values("score", ascending=False).reset_index(drop=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue["score"].value_counts())
print(queue["action"].value_counts())
queue.head(10)

score
0     154699
20     92480
80     84258
Name: count, dtype: int64
action
monitor    247179
fix_ctr     84258
Name: count, dtype: int64


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,content_type,ctr,engagement_rate,score,reason_code,action
0,client_e5c2aa26a8598242,content_d374349dbdd0ddef,82,0,6.026885,0.0,keyword article,0.000000,NaN,80,low_ctr_high_position,fix_ctr
1,client_e5c2aa26a8598242,content_d381f511a132d8e5,1376,4,9.593625,0.0,keyword article,0.002907,0.0,80,low_ctr_high_position,fix_ctr
2,client_73cda7b4e4f265ea,content_f8006d2fd362bf4e,947,0,3.351213,0.0,keyword article,0.000000,NaN,80,low_ctr_high_position,fix_ctr
3,client_73cda7b4e4f265ea,content_f7ff9b3a7b83df0d,2852,10,3.415980,0.0,keyword article,0.003506,0.0,80,low_ctr_high_position,fix_ctr
4,client_e5c2aa26a8598242,content_d396a01b0a84d480,1977,0,9.427720,0.0,keyword article,0.000000,NaN,80,low_ctr_high_position,fix_ctr
5,client_e5c2aa26a8598242,content_d3a331858dc96d41,290,1,4.899347,0.0,keyword article,0.003448,0.0,80,low_ctr_high_position,fix_ctr
6,client_2b4306c3ed003f01,content_13d2655c046f45f4,1,0,9.000000,0.0,keyword article,0.000000,NaN,80,low_ctr_high_position,fix_ctr
7,client_e5c2aa26a8598242,content_d3afb3df61a3ce6c,716,0,9.879523,0.0,keyword article,0.000000,NaN,80,low_ctr_high_position,fix_ctr
8,client_73cda7b4e4f265ea,content_f7fa41f6bfbe0abb,3901,14,3.447357,0.0,keyword article,0.003589,0.0,80,low_ctr_high_position,fix_ctr
9,client_e5c2aa26a8598242,content_d40658dbca682843,545,2,3.553483,0.0,keyword article,0.003670,0.0,80,low_ctr_high_position,fix_ctr


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*



1. fix_ctr — position 6.0, CTR 0% on 82 impressions. Wrong if: 82 impressions is too thin to trust a 0% CTR as a stable pattern.
2. fix_ctr — position 9.6, CTR 0.29% on 1,376 impressions. Wrong if: this query has a competing SERP feature (image pack, PAA) absorbing clicks.
3. fix_ctr — position 3.4, CTR 0% on 947 impressions — strong position, zero clicks is a clear red flag. Wrong if: title/snippet is broken or not rendering as expected.
4. fix_ctr — position 3.4, CTR 0.35% on 2,852 impressions — high volume makes this a high-value fix if the diagnosis holds. Wrong if: intent mismatch between the ranking query and page content.
5. fix_ctr — position 9.4, CTR 0% on 1,977 impressions. Wrong if: borderline page-1 position with natural CTR variance at this rank.
6. fix_ctr — position 4.9, CTR 0.34% on 290 impressions. Wrong if: sample is on the thinner side for a confident call.
7. fix_ctr — position 9.0, CTR 0% on 1 impression — essentially no data. Wrong if: with n=1, this is pure noise, not a real signal; should not be actioned as-is.
8. fix_ctr — position 9.9, CTR 0% on 716 impressions. Wrong if: query is highly branded/navigational, where low CTR is normal regardless of position.
9. fix_ctr — position 3.4, CTR 0.36% on 3,901 impressions — good position, solid volume, still underperforming. Wrong if: recent ranking change means this position is new/unstable.
10. fix_ctr — position 3.6, CTR 0.37% on 545 impressions. Wrong if: seasonal dip in demand for this query this month.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = queue.head(10)
top10[["client_hash_id","content_hash_id","score","reason_code","action","gsc_avg_position","ctr","gsc_impressions"]]

,client_hash_id,content_hash_id,score,reason_code,action,gsc_avg_position,ctr,gsc_impressions
0,client_e5c2aa26a8598242,content_d374349dbdd0ddef,80,low_ctr_high_position,fix_ctr,6.026885,0.000000,82
1,client_e5c2aa26a8598242,content_d381f511a132d8e5,80,low_ctr_high_position,fix_ctr,9.593625,0.002907,1376
2,client_73cda7b4e4f265ea,content_f8006d2fd362bf4e,80,low_ctr_high_position,fix_ctr,3.351213,0.000000,947
3,client_73cda7b4e4f265ea,content_f7ff9b3a7b83df0d,80,low_ctr_high_position,fix_ctr,3.415980,0.003506,2852
4,client_e5c2aa26a8598242,content_d396a01b0a84d480,80,low_ctr_high_position,fix_ctr,9.427720,0.000000,1977
5,client_e5c2aa26a8598242,content_d3a331858dc96d41,80,low_ctr_high_position,fix_ctr,4.899347,0.003448,290
6,client_2b4306c3ed003f01,content_13d2655c046f45f4,80,low_ctr_high_position,fix_ctr,9.000000,0.000000,1
7,client_e5c2aa26a8598242,content_d3afb3df61a3ce6c,80,low_ctr_high_position,fix_ctr,9.879523,0.000000,716
8,client_73cda7b4e4f265ea,content_f7fa41f6bfbe0abb,80,low_ctr_high_position,fix_ctr,3.447357,0.003589,3901
9,client_e5c2aa26a8598242,content_d40658dbca682843,80,low_ctr_high_position,fix_ctr,3.553483,0.003670,545


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: row 7 (content_13d2655c046f45f4, n=1 impression) and row 14 (content_13ad508300cafa0b, n=3 impressions, visible further down the queue) are the clearest weak picks — a single impression cannot support a confident fix_ctr recommendation, even though it mathematically satisfies the rule. This is a known limitation of a purely rule-based threshold: it doesn't account for sample size. A production version would add a minimum-impressions gate (e.g. ≥30) before flagging.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

low_n_rows = top10[top10["gsc_impressions"] < 30]
print("Top-10 rows with under 30 impressions (thin evidence):")
print(low_n_rows[["client_hash_id","content_hash_id","gsc_impressions"]])

scoring_inputs = ["gsc_impressions","gsc_clicks","gsc_avg_position","ctr"]
future_terms = ["future","next_month","is_published","is_deleted","optimized_date"]
leaked = [c for c in scoring_inputs if any(t in c.lower() for t in future_terms)]
assert len(leaked) == 0
print("Leakage check passed — rule uses only same-window, non-flag inputs.")

Top-10 rows with under 30 impressions (thin evidence):
            client_hash_id           content_hash_id  gsc_impressions
6  client_2b4306c3ed003f01  content_13d2655c046f45f4                1
Leakage check passed — rule uses only same-window, non-flag inputs.


In [20]:
import json

metrics = {
    "month": "2026-03",
    "total_rows": int(len(queue)),
    "signal_1_ctr_vs_position": {
        "verdict": "CONFIRMED",
        "buckets": {
            str(k): {"n": int(v["n"]), "avg_ctr": round(float(v["avg_ctr"]), 4)}
            for k, v in signal1_table.to_dict("index").items()
        }
    },
    "signal_2_volume_vs_engagement": {
        "verdict": "OPPOSITE",
        "buckets": {
            str(k): {"n": int(v["n"]), "avg_engagement": round(float(v["avg_engagement"]), 4)}
            for k, v in signal2_table.to_dict("index").items()
        }
    },
    "rule": {
        "reason_code": "low_ctr_high_position",
        "ctr_threshold": CTR_THRESHOLD,
        "position_threshold": 10,
        "score_distribution": queue["score"].value_counts().to_dict()
    },
    "top_10_thin_evidence_flagged": int((top10["gsc_impressions"] < 30).sum())
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w04_baseline_score_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics written to work/outputs/w04_baseline_score_metrics.json")
print(json.dumps(metrics, indent=2))

Metrics written to work/outputs/w04_baseline_score_metrics.json
{
  "month": "2026-03",
  "total_rows": 331437,
  "signal_1_ctr_vs_position": {
    "verdict": "CONFIRMED",
    "buckets": {
      "(0, 3]": {
        "n": 16144,
        "avg_ctr": 0.0106
      },
      "(3, 10]": {
        "n": 81988,
        "avg_ctr": 0.0049
      },
      "(10, 20]": {
        "n": 32203,
        "avg_ctr": 0.0032
      },
      "(20, 100]": {
        "n": 44867,
        "avg_ctr": 0.0019
      },
      "(100, 1000]": {
        "n": 102,
        "avg_ctr": 0.0061
      }
    }
  },
  "signal_2_volume_vs_engagement": {
    "verdict": "OPPOSITE",
    "buckets": {
      "(-0.001, 2.0]": {
        "n": 313,
        "avg_engagement": 0.0575
      },
      "(2.0, 216.0]": {
        "n": 9182,
        "avg_engagement": 0.0579
      },
      "(216.0, 617124.0]": {
        "n": 59342,
        "avg_engagement": 0.046
      }
    }
  },
  "rule": {
    "reason_code": "low_ctr_high_position",
    "ctr_threshold":

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.